# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading and exploring the [FAIR² Croissant dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) using the `mlcroissant` Python library.

### Dataset Source
The dataset is described using a [Croissant schema](https://mlcommons.org/croissant/). Metadata and data can be programmatically explored and loaded from the given URL.

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata and inspect basic properties
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Display available record sets, their `@id`, and associated fields with their `@id` (as defined in the Croissant schema).

In [ ]:
# The list of record sets and their ids
record_sets = list(dataset.metadata.record_sets)
print(f"Found {len(record_sets)} record set(s):\n")
for rs in record_sets:
    print(f"  Record set: {rs.name} (ID: {rs.id})")
    print("    Fields:")
    for field in rs.fields:
        print(f"      - {field.name} (@id: {field.id}, dataType: {field.data_type})")

## 3. Data Extraction
Let's load data from every record set into DataFrames for processing.
<br>
We use each record set's `@id` to reference it. You can pick specific record sets by `@id` for downstream tasks.

In [ ]:
# Collect DataFrames for each record set, referenced by @id
all_dataframes = {}
print("Loading records for each record set:\n")
for rs in record_sets:
    recs = list(dataset.records(record_set=rs.id))
    if recs:
        df = pd.DataFrame(recs)
        all_dataframes[rs.id] = df
        print(f"  Loaded {len(df)} records for record set '{rs.name}' (@id: {rs.id}) with columns: {list(df.columns)}")
    else:
        print(f"  No records loaded for record set '{rs.name}' (@id: {rs.id})")

# For demonstration pick the first available record set
if all_dataframes:
    first_rs_id = list(all_dataframes.keys())[0]
    print(f"\nFirst record set DataFrame (ID: {first_rs_id}) preview:")
    display(all_dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
We explore the content of the first available record set. Replace the used field IDs with those best suited for your investigation.

We: 
- Select a numeric field by `@id` (see previous 'fields' list) and filter the dataset for numeric values above a threshold.
- Normalize that field.
- Group by a categorical field if present.

In [ ]:
if all_dataframes:
    rs = next(r for r in record_sets if r.id == first_rs_id)
    # Find numeric fields (with data_type containing 'Float' or 'Integer')
    numeric_field_obj = next((f for f in rs.fields if f.data_type and ("Float" in f.data_type or "Integer" in f.data_type)), None)
    if numeric_field_obj:
        numeric_field_id = numeric_field_obj.id
        print(f"Using numeric field: '{numeric_field_obj.name}' (@id: {numeric_field_id})")
        df = all_dataframes[first_rs_id]
        # Sometimes field IDs are used as DataFrame column names; fallback to field name if necessary
        col = numeric_field_id if numeric_field_id in df.columns else numeric_field_obj.name if numeric_field_obj.name in df.columns else None
        if col:
            # Coerce to numeric
            df[col] = pd.to_numeric(df[col], errors='coerce')
            threshold = df[col].mean() if not pd.isna(df[col].mean()) else 10
            filtered_df = df[df[col] > threshold]
            print(f"\nFiltered records with '{col}' > {threshold:.3f} (records={len(filtered_df)}):")
            display(filtered_df.head())
            
            # Normalize column
            norm_col = f"{col}_normalized"
            filtered_df[norm_col] = (filtered_df[col] - filtered_df[col].mean()) / filtered_df[col].std()
            print(f"\nNormalized '{col}' for filtered records:")
            display(filtered_df[[col, norm_col]].head())

            # Try to group by a categorical field for mean analysis
            group_field_obj = next((f for f in rs.fields if f.data_type == 'Text'), None)
            if group_field_obj:
                group_col = group_field_obj.id if group_field_obj.id in df.columns else group_field_obj.name
                if group_col in df.columns:
                    grouped_df = filtered_df.groupby(group_col)[col].mean().to_frame(name=f"mean_{col}")
                    print(f"\nGrouped filtered data by '{group_col}':")
                    display(grouped_df.head())
    else:
        print("No numeric field found in the first record set.")

## 5. Visualization
Visualize the distribution of the numeric field in the record set (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if all_dataframes and numeric_field_obj and col and col in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[col].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of field '{col}' in record set '{first_rs_id}'")
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion

In this notebook, we used `mlcroissant` to:
- Programmatically access rich metadata and data from the FAIR² dataset described by a Croissant schema
- Explore record sets and field structures using their canonical `@id`
- Extract, filter, and normalize numeric data, and group by categorical fields
- Visualize field distributions

You can now extend this analysis to other record sets/fields, or further data science and machine learning tasks, always referencing entities by `@id` for reproducibility and metadata-traceable workflows.